<a href="https://colab.research.google.com/github/jayyyyqwq/resheightnet/blob/main/notebooks/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResHeightNet — Colab training (M7)

Run cells top to bottom. The GPU assert in the next cell is deliberately
**first** and runs before any download — free-tier Colab can silently
hand you a CPU-only runtime, and discovering that after a 13.85 GB
download wastes the whole session.

Data lives on `/content` (local disk), never on a Drive mount — Drive
FUSE is far too slow to train from (~9.6 min/epoch of I/O vs ~29s from
local disk), and the 13.85 GB dataset doesn't fit in free Drive's 15 GB
quota alongside checkpoints anyway. Only checkpoints go to Drive.

In [1]:
import torch
assert torch.cuda.is_available(), (
    "No GPU available -- change Runtime > Change runtime type > T4 GPU, "
    "or your GPU quota is exhausted. ABORT before downloading 13.85 GB."
)
print(torch.__version__, torch.cuda.get_device_name(0))
assert "T4" in torch.cuda.get_device_name(0), (
    "Not a T4 -- the timing estimates in the execution plan assume a T4; "
    "re-check them on a different GPU before trusting elapsed-time output."
)

2.11.0+cu128 Tesla T4


In [2]:
# Do NOT pin torch/torchvision here -- use Colab's preinstalled CUDA
# build. Pinning to 2.6.0 downloads ~2.5GB, can silently land a
# CPU-only wheel, forces a runtime restart, and desyncs torchvision.
!pip install -q h5py huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 56.8 MB/s eta 0:00:00


In [3]:
# The repo carries data/splits/*.txt committed, so this step alone is
# enough to fetch the deterministic split manifests -- no dependency
# on the local DepthWizard checkout.
!git clone https://github.com/jayyyyqwq/resheightnet.git /content/resheightnet
%cd /content/resheightnet

Cloning into '/content/resheightnet'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 51 (delta 3), reused 51 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 85.10 KiB | 829.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/resheightnet


In [4]:
import os
os.environ["GAMUS_ROOT"] = "/content/gamus"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

!python scripts/fetch_gamus.py --split data/splits/stage_a1_train.txt --split data/splits/stage_a1_val.txt --data-root /content/gamus

Streaming output truncated to the last 5000 lines.
heights/val/PHL_6524_AGL.h5: downloading bytes: 100% 3.58M/3.58M [00:00<00:00, 5.92MB/s,  353kB/s  ]
heights/val/PHL_6524_AGL.h5: reconstructing file: 100% 4.20M/4.20M [00:00<00:00, 6.93MB/s,  414kB/s  ]

heights/val/PHL_6544_AGL.h5: downloading bytes:   0% 0.00/4.20M [00:00<?, ?B/s]
heights/val/PHL_6547_AGL.h5: downloading bytes:   0% 0.00/4.20M [00:00<?, ?B/s]
heights/val/PHL_6554_AGL.h5: downloading bytes:   0% 0.00/4.20M [00:00<?, ?B/s]
heights/val/PHL_6555_AGL.h5: downloading bytes:   0% 0.00/4.20M [00:00<?, ?B/s]
heights/val/PHL_6531_AGL.h5: downloading bytes: 100% 3.52M/3.52M [00:00<00:00, 4.52MB/s,  346kB/s  ]
heights/val/PHL_6531_AGL.h5: reconstructing file: 100% 4.20M/4.20M [00:00<00:00, 5.39MB/s,  412kB/s  ]

heights/val/PHL_6532_AGL.h5: downloading bytes:   6% 248k/4.20M [00:00<00:09, 431kB/s]
heights/val/PHL_6532_AGL.h5: downloading bytes: 100% 3.67M/3.67M [00:00<00:00, 5.87MB/s,  362kB/s  ]
heights/val/PHL_6532_AGL.h5: re

In [5]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_CKPT_DIR = "/content/drive/MyDrive/resheightnet/checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# Write checkpoints to local /content first (fast); train_full already
# does this via --checkpoint-dir. We copy to Drive after each run below
# rather than pointing --checkpoint-dir at Drive directly, since direct
# mid-epoch Drive writes stall.
LOCAL_CKPT_DIR = "/content/ckpt"

Mounted at /content/drive


In [ ]:
# M6 overfit gate -- run this BEFORE the full training run. If it
# fails, do not proceed to the cell below; see src/train.py's failure
# taxonomy docstring for how to diagnose which symptom you're seeing.
!python -m src.train --overfit 10 --steps 800 --val-split data/splits/stage_a1_val.txt --data-root /content/gamus

step 0000  loss=5.3937 m  (L0=5.3066 m)
step 0010  loss=4.5173 m  (L0=5.3066 m)
step 0020  loss=3.1944 m  (L0=5.3066 m)
step 0030  loss=1.8965 m  (L0=5.3066 m)
step 0040  loss=1.6585 m  (L0=5.3066 m)
step 0050  loss=1.3781 m  (L0=5.3066 m)
step 0060  loss=1.2572 m  (L0=5.3066 m)
step 0070  loss=1.2030 m  (L0=5.3066 m)
step 0080  loss=1.0658 m  (L0=5.3066 m)
step 0090  loss=0.9980 m  (L0=5.3066 m)
step 0100  loss=0.9190 m  (L0=5.3066 m)
step 0110  loss=0.9355 m  (L0=5.3066 m)
step 0120  loss=0.8191 m  (L0=5.3066 m)
step 0130  loss=0.7804 m  (L0=5.3066 m)
step 0140  loss=0.7303 m  (L0=5.3066 m)
step 0150  loss=0.7299 m  (L0=5.3066 m)
step 0160  loss=0.6759 m  (L0=5.3066 m)
step 0170  loss=0.6625 m  (L0=5.3066 m)
step 0180  loss=0.6506 m  (L0=5.3066 m)
step 0190  loss=0.6701 m  (L0=5.3066 m)
step 0200  loss=0.5796 m  (L0=5.3066 m)
step 0210  loss=0.5808 m  (L0=5.3066 m)
step 0220  loss=0.5683 m  (L0=5.3066 m)
step 0230  loss=0.5157 m  (L0=5.3066 m)
step 0240  loss=0.5069 m  (L0=5.3066 m)


In [7]:
# Full training: 15 epochs, matching DepthWizard's recipe exactly
# (AdamW lr=1e-4 wd=1e-4, CosineAnnealingLR, seed=42, batch=8 @ 512x512).
# Pass --resume-from $LOCAL_CKPT_DIR/last.pt to continue after a
# disconnect (resume correctness is bitwise-verified locally in
# tests/test_integration.py::test_resume_is_bitwise_identical_to_uninterrupted_training).
!python -m src.train \
  --data-root /content/gamus \
  --epochs 15 --batch-size 8 --patch-size 512 \
  --lr 1e-4 --weight-decay 1e-4 --seed 42 \
  --num-workers 2 \
  --checkpoint-dir {LOCAL_CKPT_DIR}

epoch 01/15  train_L1=4.8213 m  val_L1=4.8312 m  (63.0s)
  [checkpoint] new best val_L1=4.8312 m -> /content/ckpt/best.pt
epoch 02/15  train_L1=4.3243 m  val_L1=4.0684 m  (59.4s)
  [checkpoint] new best val_L1=4.0684 m -> /content/ckpt/best.pt
epoch 03/15  train_L1=nan m  val_L1=3.9739 m  (59.8s)
  [checkpoint] new best val_L1=3.9739 m -> /content/ckpt/best.pt
Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x79562e038c20>
Traceback (most recent call last):
  File "/usr/lib/python3.13/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/resheightnet/src/train.py", line 355, in <module>
    main()
    ~~~~^^
  File "/content/resheightnet/src/train.py", line 337, in main
    train_full(
    ~~~~~~~~~~^
        train_split=args

In [8]:
# Copy checkpoints to Drive so they survive the session.
!cp {LOCAL_CKPT_DIR}/last.pt {LOCAL_CKPT_DIR}/best.pt "{DRIVE_CKPT_DIR}/"

In [9]:
# M8: evaluate at the DepthWizard-parity protocol (512 center crop,
# pooled per-pixel, unclamped predictions) and write results for
# docs/comparison_table.md.
!python -m src.evaluate \
  --split data/splits/stage_a1_val.txt \
  --data-root /content/gamus \
  --checkpoint {LOCAL_CKPT_DIR}/best.pt \
  --out /content/resheightnet_metrics.json

!cp /content/resheightnet_metrics.json "{DRIVE_CKPT_DIR}/../resheightnet_metrics.json"

{
  "mae_m": 3.9564385414123535,
  "rmse_m": 7.864644497427537,
  "pearson_r": 0.5500645881412043,
  "spearman_rho": 0.6055774119529842,
  "n_valid": 51283929,
  "n_tiles": 200,
  "protocol": "512_center_crop_pooled",
  "pred_finite_coverage": 1.0
}
class_mae: {
  "Others": 1.3637498617172241,
  "Ground": 0.341459184885025,
  "Low vegetation": 0.4663415551185608,
  "Buildings": 6.074038505554199,
  "Water": 0.35185933113098145,
  "Road": 1.8107538223266602,
  "Tree": 9.261320114135742
}
bucket_mae: {
  "0-2m": 0.2729746997356415,
  "2-10m": 3.6909430027008057,
  "10-20m": 11.06467056274414,
  "20-50m": 24.529499053955078,
  ">50m": 68.10153198242188
}
Wrote metrics to /content/resheightnet_metrics.json
